# Fake News Prediction Rebuild — Step 2: EDA

Goal: understand the dataset honestly before writing any feature/model code, and specifically check for leakage before it costs us a fake 99% accuracy number later.

This dataset (`Fake.csv` / `True.csv`, ISOT-style) has no `author` column, so the leakage story is different from the old notebook's author-leakage problem. Here the suspects are:
- `Subject` — the two files were likely scraped from different sections of the source site, so `Subject` might separate real/fake almost perfectly without the model learning anything about *content*.
- Reuters dateline pattern (`WASHINGTON (Reuters) -` style) at the start of `text` in the true-news file, which would let a model key off boilerplate formatting instead of substance.

Both are checked below with real numbers, not assumed.

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_colwidth', 120)

## 1. Load, label, concat, shuffle

The two files come in solid blocks (all-fake rows, then all-true rows). If we don't shuffle before any order-sensitive step (train/test split, head() sanity checks, cross-validation folds), we risk correlated splits. Shuffle once here, right after labeling, with a fixed `random_state` so the notebook is reproducible.

In [ ]:
RANDOM_STATE = 42

fake_df = pd.read_csv('../data/Fake.csv')
true_df = pd.read_csv('../data/True.csv')

fake_df['label'] = 1  # 1 = fake, matches the old notebook's convention
true_df['label'] = 0  # 0 = real

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(df.shape)
df.head()

## 2. Class balance and missing values

In [ ]:
print("Raw counts:")
print(df['label'].value_counts())
print("\nNormalized:")
print(df['label'].value_counts(normalize=True).round(4))

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'pct': missing_pct})

## 3. Subject vs label — checking for leakage

If any `Subject` value maps almost 100% to one label, that column is a shortcut, not a feature. A model trained on it would look great on this dataset and be useless on a real newsroom feed, because "subject tag from this specific scrape" isn't a property of real vs fake content in general.

In [ ]:
subject_crosstab = pd.crosstab(df['subject'], df['label'], normalize='index').round(4)
subject_crosstab.columns = ['pct_real', 'pct_fake']
subject_counts = df['subject'].value_counts()
subject_summary = subject_crosstab.join(subject_counts.rename('n'))
subject_summary.sort_values('n', ascending=False)

**Read this table before moving on.** If any subject is >95% one label, treat `Subject` as excluded-by-design for modeling, and say so explicitly in `src/preprocess.py` and the README rather than silently dropping the column.

## 4. Reuters dateline check

First confirm the pattern actually matches real rows before drawing any conclusion from it — don't assume it holds just because it's a known pattern for this dataset.

In [ ]:
reuters_pattern = re.compile(r'^\s*\S.*?\(Reuters\)\s*-')

df['has_reuters_dateline'] = df['text'].fillna('').apply(lambda t: bool(reuters_pattern.match(t)))

print("Sample matches:")
for t in df.loc[df['has_reuters_dateline'], 'text'].head(3):
    print(t[:100])
    print('---')

In [ ]:
dateline_crosstab = pd.crosstab(df['has_reuters_dateline'], df['label'], normalize='index').round(4)
dateline_crosstab.columns = ['pct_real', 'pct_fake']
dateline_crosstab

If this shows the dateline appears almost exclusively in real-labeled rows, strip it before cleaning in Step 3 (regex substitution at the start of `text`, done once in `src/preprocess.py`), otherwise the model just learns to detect "(Reuters)" instead of anything about real vs fake content.

## 5. Date range check

In [ ]:
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')

unparseable = df['date_parsed'].isnull().sum()
print(f"Unparseable dates: {unparseable} ({unparseable / len(df) * 100:.2f}%)")

print("\nDate range by label:")
print(df.groupby('label')['date_parsed'].agg(['min', 'max', 'count']))

Not used as a model feature either way (a raw date shouldn't predict truthfulness), but worth knowing if the ranges are disjoint, since that would be one more thing that quietly makes this dataset easier than real-world detection would be.

## 6. Summary

Fill this in after running the cells above with the real dataset. Do not write conclusions here before the numbers are in front of you.

- Class balance: `<fill in>`
- Missing values: `<fill in>`
- Subject leakage: `<fill in — which subjects, what pct, decision on exclusion>`
- Reuters dateline leakage: `<fill in — pct match, decision on stripping>`
- Date range: `<fill in>`